# 25 — Semisupervised Feature Space

## Objetivo

Determinar qué espacio de características puede utilizarse de forma científicamente defendible para integrar datos etiquetados `L` y no etiquetados `U`.

Este notebook distingue dos niveles:

1. **STRICT_CANONICAL**  
   Solo variables con equivalencia empírica suficientemente fuerte entre una fuente etiquetada y una fuente no etiquetada.

2. **EXPERIMENTAL_CONSTRUCT_PROXY**  
   Variables estructurales genéricas que pueden calcularse en más de una modalidad, pero cuya equivalencia es de constructo y no exacta.

No se entrenará ningún modelo aquí.

La pregunta que debe responder el notebook es:

> ¿Existe actualmente un `X_common` que pueda compartirse entre datos Nielsen etiquetados, datos no etiquetados y la futura extensión?

Si la respuesta estricta es no, se registrará como resultado metodológico y se conservará un espacio proxy únicamente para experimentos posteriores.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = PROJECT_DIR / "reports" / "tables"
PUBLIC_INTERIM_DIR = PROJECT_DIR / "data" / "public_v2" / "interim"

SEMISUP_DIR = PROJECT_DIR / "data" / "semisupervised"
SEMISUP_REPORTS_DIR = REPORTS_DIR / "semisupervised"
SEMISUP_INTERIM_DIR = SEMISUP_DIR / "interim"

SEMISUP_REPORTS_DIR.mkdir(parents=True, exist_ok=True)
SEMISUP_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)


PROJECT_DIR: /Users/uriel/Documents/machine_learning


## 2. Funciones de localización y carga

In [2]:
SEARCH_ROOTS = [
    PUBLIC_INTERIM_DIR,
    REPORTS_DIR
]


def locate_file(filename):
    matches = []

    for root in SEARCH_ROOTS:
        if not root.exists():
            continue

        direct = root / filename
        if direct.exists():
            matches.append(direct)

        for path in root.rglob(filename):
            if path.exists():
                matches.append(path)

    unique = []
    seen = set()

    for path in matches:
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(path)

    if not unique:
        return None

    # Prefer public_v2/interim for row-level integrated datasets.
    unique = sorted(
        unique,
        key=lambda p: (
            0 if PUBLIC_INTERIM_DIR in p.parents else 1,
            len(str(p))
        )
    )

    return unique[0]


def load_csv(filename, required=False):
    path = locate_file(filename)

    if path is None:
        if required:
            raise FileNotFoundError(
                f"No se encontró {filename}"
            )

        print("NO ENCONTRADO:", filename)
        return None, None

    df = pd.read_csv(path)

    print(
        filename,
        "->",
        path.relative_to(PROJECT_DIR),
        df.shape
    )

    return df, path


## 3. Cargar artefactos principales

In [3]:
canonical_x_seed, canonical_path = load_csv(
    "semisupervised_canonical_x_seed.csv",
    required=True
)

compatibility, compatibility_path = load_csv(
    "feature_compatibility_matrix.csv",
    required=True
)

catching, catching_path = load_csv(
    "catching_static_validation_dataset.csv",
    required=True
)

rico, rico_path = load_csv(
    "uicrit_rico_proxy_dataset.csv",
    required=True
)

dashboard_map, dashboard_map_path = load_csv(
    "dashboards_feature_map.csv",
    required=False
)

direct_label_audit, direct_label_audit_path = load_csv(
    "direct_label_class_audit_final.csv",
    required=False
)

print("\nCanonical X:", canonical_x_seed.shape)
print("Compatibility:", compatibility.shape)
print("Catching:", catching.shape)
print("UICrit/RICO:", rico.shape)


semisupervised_canonical_x_seed.csv -> reports/tables/semisupervised/semisupervised_canonical_x_seed.csv (40, 10)
feature_compatibility_matrix.csv -> reports/tables/feature_compatibility_matrix.csv (80, 8)
catching_static_validation_dataset.csv -> data/public_v2/interim/catching_static_validation_dataset.csv (2, 23)
uicrit_rico_proxy_dataset.csv -> data/public_v2/interim/uicrit_rico_proxy_dataset.csv (1000, 23)
dashboards_feature_map.csv -> reports/tables/dashboards_feature_map.csv (11, 4)
direct_label_class_audit_final.csv -> reports/tables/direct_label_class_audit_final.csv (8, 10)

Canonical X: (40, 10)
Compatibility: (80, 8)
Catching: (2, 23)
UICrit/RICO: (1000, 23)


### Nota sobre `direct_label_class_audit_final.csv`

El notebook 24 podía marcar este archivo como `found=False` porque su inventario buscaba patrones temáticos y no incluía `direct_label*`.

Aquí se comprueba directamente por nombre. Su ausencia real y su ausencia del inventario son dos situaciones diferentes.


## 4. Auditar columnas reales de las fuentes

In [4]:
schema_audit = pd.DataFrame([
    {
        "source_dataset": "Catching_UX_Flaws",
        "rows": len(catching),
        "columns": len(catching.columns),
        "column_names": " | ".join(catching.columns.astype(str))
    },
    {
        "source_dataset": "UICrit_RICO",
        "rows": len(rico),
        "columns": len(rico.columns),
        "column_names": " | ".join(rico.columns.astype(str))
    }
])

display(schema_audit)

print("\nCATCHING COLUMNS")
print(catching.columns.tolist())

print("\nRICO COLUMNS")
print(rico.columns.tolist())


,source_dataset,rows,columns,column_names
0,Catching_UX_Flaws,2,23,site | repo_full_name | static_page_count | static_mean_element_count | static_max_element_count | static_mean_dom_depth | static_max_dom_depth | static_total_interactive_controls | static_control_label_proxy_ratio | static_total_form_controls | static_native_validation_ratio | static_mean_class_reuse_ratio | commit_sha | label_source | label_level | H1_violation | H4_violation | H5_violation | H6_violation | source_dataset | feature_modality | primary_training | intended_use
1,UICrit_RICO,1000,23,rico_id | rico_parse_ok | rico_node_count | rico_visible_node_count | rico_leaf_count | rico_max_tree_depth | rico_mean_branching_factor | rico_interactive_node_count | rico_scrollable_node_count | rico_resource_id_ratio | rico_interactive_label_proxy_ratio | rico_unlabeled_interactive_count | rico_unique_class_count | rico_class_diversity_ratio | proxy_record_count | learnability_proxy | design_quality_proxy | learnability_proxy_mean | design_quality_proxy_mean | source_dataset | label_leve...



CATCHING COLUMNS
['site', 'repo_full_name', 'static_page_count', 'static_mean_element_count', 'static_max_element_count', 'static_mean_dom_depth', 'static_max_dom_depth', 'static_total_interactive_controls', 'static_control_label_proxy_ratio', 'static_total_form_controls', 'static_native_validation_ratio', 'static_mean_class_reuse_ratio', 'commit_sha', 'label_source', 'label_level', 'H1_violation', 'H4_violation', 'H5_violation', 'H6_violation', 'source_dataset', 'feature_modality', 'primary_training', 'intended_use']

RICO COLUMNS
['rico_id', 'rico_parse_ok', 'rico_node_count', 'rico_visible_node_count', 'rico_leaf_count', 'rico_max_tree_depth', 'rico_mean_branching_factor', 'rico_interactive_node_count', 'rico_scrollable_node_count', 'rico_resource_id_ratio', 'rico_interactive_label_proxy_ratio', 'rico_unlabeled_interactive_count', 'rico_unique_class_count', 'rico_class_diversity_ratio', 'proxy_record_count', 'learnability_proxy', 'design_quality_proxy', 'learnability_proxy_mean', '

## 5. Auditar matriz de compatibilidad existente

In [5]:
print(
    "Compatibility columns:",
    compatibility.columns.tolist()
)

display(
    compatibility.head(30)
)


Compatibility columns: ['source_dataset', 'canonical_variable', 'source_variable', 'compatibility_level', 'possible_heuristic', 'reason', 'can_be_renamed_canonical', 'can_share_feature_space']


,source_dataset,canonical_variable,source_variable,compatibility_level,possible_heuristic,reason,can_be_renamed_canonical,can_share_feature_space
0,UICrit_RICO,task_time_s,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
1,UICrit_RICO,mean_response_time_ms,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
2,UICrit_RICO,max_response_time_ms,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
3,UICrit_RICO,mean_pre_action_pause_s,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
4,UICrit_RICO,click_count,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
5,UICrit_RICO,keypress_count,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
6,UICrit_RICO,scroll_count,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
7,UICrit_RICO,backtracking_count,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
8,UICrit_RICO,repeated_action_count,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
9,UICrit_RICO,navigation_steps,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False


In [6]:
def detect_status_column(df):
    tokens = {
        "EXACT",
        "CONSTRUCT_PROXY",
        "PARTIAL",
        "NO_MAPPING"
    }

    best_col = None
    best_hits = 0

    for col in df.columns:
        values = (
            df[col]
            .dropna()
            .astype(str)
            .str.upper()
        )

        hits = int(
            values.apply(
                lambda v: any(
                    token in v
                    for token in tokens
                )
            ).sum()
        )

        if hits > best_hits:
            best_hits = hits
            best_col = col

    return best_col, best_hits


status_col, status_hits = detect_status_column(
    compatibility
)

print(
    "Status column detectada:",
    status_col,
    "| hits:",
    status_hits
)

if status_col is not None:
    print(
        compatibility[
            status_col
        ]
        .value_counts(
            dropna=False
        )
    )


Status column detectada: compatibility_level | hits: 80
compatibility_level
NO_MAPPING         69
CONSTRUCT_PROXY    10
PARTIAL             1
Name: count, dtype: int64


## 6. Resultado estricto heredado de la compatibilidad

Para el espacio `STRICT_CANONICAL`, solo se admitirán estados equivalentes a:

- `EXACT`
- `DERIVABLE_EXACT`

`CONSTRUCT_PROXY` y `PARTIAL` quedan fuera del modelo primario.


In [7]:
STRICT_STATUSES = {
    "EXACT",
    "DERIVABLE_EXACT"
}

if status_col is None:
    strict_rows = pd.DataFrame(
        columns=compatibility.columns
    )

else:
    normalized_status = (
        compatibility[
            status_col
        ]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.strip()
    )

    strict_rows = compatibility[
        normalized_status.isin(
            STRICT_STATUSES
        )
    ].copy()

print(
    "Filas STRICT encontradas:",
    len(strict_rows)
)

display(
    strict_rows
)


Filas STRICT encontradas: 0


,source_dataset,canonical_variable,source_variable,compatibility_level,possible_heuristic,reason,can_be_renamed_canonical,can_share_feature_space


## Corrección aplicada

Se corrigió el nombre de la variable de UICrit/RICO usada para el constructo
`interactive_control_count`:

```text
incorrecto: rico_interactive_count
correcto:   rico_interactive_node_count
```

La columna correcta está presente en `uicrit_rico_proxy_dataset.csv`.

Esta corrección afecta la auditoría de disponibilidad/cobertura del espacio
`EXPERIMENTAL_CONSTRUCT_PROXY`. No modifica el resultado heredado de
`STRICT_CANONICAL`: ese espacio sigue dependiendo exclusivamente de la matriz
de compatibilidad y debe volver a calcularse al ejecutar el notebook.


## 7. Espacio estructural proxy candidato

Como alternativa experimental, se evalúan tres constructos estructurales genéricos que pueden ser calculados tanto en HTML/DOM web como en una jerarquía de UI:

- profundidad de la estructura;
- cantidad de controles interactivos;
- proporción de controles interactivos con señalización textual.

La equivalencia entre Web DOM y Android UI hierarchy **no es exacta**. Por tanto, estos features se marcan explícitamente como `EXPERIMENTAL_CONSTRUCT_PROXY`.


In [8]:
proxy_mapping_declared = pd.DataFrame([
    {
        "model_feature": "structure_depth",
        "source_dataset": "Catching_UX_Flaws",
        "source_column": "static_max_dom_depth",
        "transform": "identity_numeric",
        "mapping_level": "CONSTRUCT_PROXY",
        "feature_definition": "Maximum structural tree depth of the evaluated interface aggregate.",
        "extension_derivable": True
    },
    {
        "model_feature": "structure_depth",
        "source_dataset": "UICrit_RICO",
        "source_column": "rico_max_tree_depth",
        "transform": "identity_numeric",
        "mapping_level": "CONSTRUCT_PROXY",
        "feature_definition": "Maximum structural tree depth of the evaluated interface aggregate.",
        "extension_derivable": True
    },
    {
        "model_feature": "interactive_control_count",
        "source_dataset": "Catching_UX_Flaws",
        "source_column": "static_total_interactive_controls",
        "transform": "identity_numeric",
        "mapping_level": "CONSTRUCT_PROXY",
        "feature_definition": "Count of interactive controls in the evaluated interface aggregate.",
        "extension_derivable": True
    },
    {
        "model_feature": "interactive_control_count",
        "source_dataset": "UICrit_RICO",
        "source_column": "rico_interactive_node_count",
        "transform": "identity_numeric",
        "mapping_level": "CONSTRUCT_PROXY",
        "feature_definition": "Count of interactive controls in the evaluated interface aggregate.",
        "extension_derivable": True
    },
    {
        "model_feature": "interactive_label_ratio",
        "source_dataset": "Catching_UX_Flaws",
        "source_column": "static_control_label_proxy_ratio",
        "transform": "identity_numeric",
        "mapping_level": "CONSTRUCT_PROXY",
        "feature_definition": "Fraction of interactive controls with an observable textual/label proxy.",
        "extension_derivable": True
    },
    {
        "model_feature": "interactive_label_ratio",
        "source_dataset": "UICrit_RICO",
        "source_column": "rico_interactive_label_proxy_ratio",
        "transform": "identity_numeric",
        "mapping_level": "CONSTRUCT_PROXY",
        "feature_definition": "Fraction of interactive controls with an observable textual/label proxy.",
        "extension_derivable": True
    }
])

display(
    proxy_mapping_declared
)


,model_feature,source_dataset,source_column,transform,mapping_level,feature_definition,extension_derivable
0,structure_depth,Catching_UX_Flaws,static_max_dom_depth,identity_numeric,CONSTRUCT_PROXY,Maximum structural tree depth of the evaluated interface aggregate.,True
1,structure_depth,UICrit_RICO,rico_max_tree_depth,identity_numeric,CONSTRUCT_PROXY,Maximum structural tree depth of the evaluated interface aggregate.,True
2,interactive_control_count,Catching_UX_Flaws,static_total_interactive_controls,identity_numeric,CONSTRUCT_PROXY,Count of interactive controls in the evaluated interface aggregate.,True
3,interactive_control_count,UICrit_RICO,rico_interactive_node_count,identity_numeric,CONSTRUCT_PROXY,Count of interactive controls in the evaluated interface aggregate.,True
4,interactive_label_ratio,Catching_UX_Flaws,static_control_label_proxy_ratio,identity_numeric,CONSTRUCT_PROXY,Fraction of interactive controls with an observable textual/label proxy.,True
5,interactive_label_ratio,UICrit_RICO,rico_interactive_label_proxy_ratio,identity_numeric,CONSTRUCT_PROXY,Fraction of interactive controls with an observable textual/label proxy.,True


## 8. Verificar que los mappings existen realmente

In [9]:
source_frames = {
    "Catching_UX_Flaws": catching,
    "UICrit_RICO": rico
}

mapping_checks = []

for _, row in proxy_mapping_declared.iterrows():
    source = row["source_dataset"]
    column = row["source_column"]

    df = source_frames.get(source)

    column_exists = (
        df is not None
        and column in df.columns
    )

    if column_exists:
        values = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        n_valid = int(
            values.notna().sum()
        )

        n_unique = int(
            values.dropna().nunique()
        )

        min_value = (
            float(values.min())
            if n_valid > 0
            else np.nan
        )

        max_value = (
            float(values.max())
            if n_valid > 0
            else np.nan
        )

    else:
        n_valid = 0
        n_unique = 0
        min_value = np.nan
        max_value = np.nan

    mapping_checks.append({
        **row.to_dict(),
        "column_exists": column_exists,
        "n_valid": n_valid,
        "n_unique": n_unique,
        "min_value": min_value,
        "max_value": max_value
    })

proxy_mapping_audit = pd.DataFrame(
    mapping_checks
)

display(
    proxy_mapping_audit
)


,model_feature,source_dataset,source_column,transform,mapping_level,feature_definition,extension_derivable,column_exists,n_valid,n_unique,min_value,max_value
0,structure_depth,Catching_UX_Flaws,static_max_dom_depth,identity_numeric,CONSTRUCT_PROXY,Maximum structural tree depth of the evaluated interface aggregate.,True,True,2,2,8.00000,15.000000
1,structure_depth,UICrit_RICO,rico_max_tree_depth,identity_numeric,CONSTRUCT_PROXY,Maximum structural tree depth of the evaluated interface aggregate.,True,True,1000,22,4.00000,25.000000
2,interactive_control_count,Catching_UX_Flaws,static_total_interactive_controls,identity_numeric,CONSTRUCT_PROXY,Count of interactive controls in the evaluated interface aggregate.,True,True,2,2,23.00000,585.000000
3,interactive_control_count,UICrit_RICO,rico_interactive_node_count,identity_numeric,CONSTRUCT_PROXY,Count of interactive controls in the evaluated interface aggregate.,True,True,1000,52,0.00000,80.000000
4,interactive_label_ratio,Catching_UX_Flaws,static_control_label_proxy_ratio,identity_numeric,CONSTRUCT_PROXY,Fraction of interactive controls with an observable textual/label proxy.,True,True,2,2,0.73913,0.842735
5,interactive_label_ratio,UICrit_RICO,rico_interactive_label_proxy_ratio,identity_numeric,CONSTRUCT_PROXY,Fraction of interactive controls with an observable textual/label proxy.,True,True,998,139,0.00000,1.000000


## 9. Validaciones semánticas/rango del espacio proxy

Estas validaciones no demuestran equivalencia entre plataformas. Solo comprueban que los datos pueden representarse bajo el constructo declarado.


In [10]:
proxy_validation_records = []

for feature in [
    "structure_depth",
    "interactive_control_count",
    "interactive_label_ratio"
]:
    subset = proxy_mapping_audit[
        proxy_mapping_audit[
            "model_feature"
        ]
        ==
        feature
    ]

    all_columns_exist = bool(
        len(subset) >= 2
        and subset[
            "column_exists"
        ].all()
    )

    both_sources_have_data = bool(
        len(subset) >= 2
        and (
            subset[
                "n_valid"
            ]
            >
            0
        ).all()
    )

    proxy_validation_records.append({
        "model_feature": feature,
        "all_columns_exist": all_columns_exist,
        "both_sources_have_data": both_sources_have_data,
        "proxy_feature_usable": (
            all_columns_exist
            and both_sources_have_data
        )
    })

proxy_feature_validation = pd.DataFrame(
    proxy_validation_records
)

display(
    proxy_feature_validation
)


,model_feature,all_columns_exist,both_sources_have_data,proxy_feature_usable
0,structure_depth,True,True,True
1,interactive_control_count,True,True,True
2,interactive_label_ratio,True,True,True


## 10. Construir registro de espacios de features

In [11]:
strict_feature_names = []

# La matriz histórica es fuente de evidencia de compatibilidad.
# Solo se agregan nombres si podemos identificar una columna de variable canónica.
canonical_name_candidates = [
    col
    for col in compatibility.columns
    if (
        "canonical" in col.lower()
        or
        "variable" in col.lower()
        or
        "feature" in col.lower()
    )
]

if (
    len(strict_rows) > 0
    and len(canonical_name_candidates) > 0
):
    candidate_col = canonical_name_candidates[0]

    strict_feature_names = sorted(
        strict_rows[
            candidate_col
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

usable_proxy_features = (
    proxy_feature_validation.loc[
        proxy_feature_validation[
            "proxy_feature_usable"
        ],
        "model_feature"
    ]
    .tolist()
)

feature_space_registry = pd.DataFrame([
    {
        "feature_space": "STRICT_CANONICAL",
        "n_features": len(strict_feature_names),
        "features": " | ".join(strict_feature_names),
        "scientific_role": "PRIMARY_CANDIDATE",
        "cross_platform_equivalence": "REQUIRES_EXACT_OR_DERIVABLE_EXACT",
        "extension_derivable": True,
        "status": (
            "AVAILABLE"
            if len(strict_feature_names) > 0
            else "EMPTY"
        )
    },
    {
        "feature_space": "EXPERIMENTAL_CONSTRUCT_PROXY",
        "n_features": len(usable_proxy_features),
        "features": " | ".join(usable_proxy_features),
        "scientific_role": "SECONDARY_EXPERIMENT_ONLY",
        "cross_platform_equivalence": "CONSTRUCT_LEVEL_ONLY",
        "extension_derivable": True,
        "status": (
            "AVAILABLE"
            if len(usable_proxy_features) > 0
            else "EMPTY"
        )
    }
])

display(
    feature_space_registry
)


,feature_space,n_features,features,scientific_role,cross_platform_equivalence,extension_derivable,status
0,STRICT_CANONICAL,0,,PRIMARY_CANDIDATE,REQUIRES_EXACT_OR_DERIVABLE_EXACT,True,EMPTY
1,EXPERIMENTAL_CONSTRUCT_PROXY,3,structure_depth | interactive_control_count | interactive_label_ratio,SECONDARY_EXPERIMENT_ONLY,CONSTRUCT_LEVEL_ONLY,True,AVAILABLE


## 11. Cobertura por fuente

La cobertura se calcula sobre el espacio proxy declarado.  
No se imputa ningún valor faltante.


In [12]:
coverage_records = []

for source, df in source_frames.items():
    source_maps = proxy_mapping_audit[
        proxy_mapping_audit[
            "source_dataset"
        ]
        ==
        source
    ]

    for _, mapping in source_maps.iterrows():
        feature = mapping["model_feature"]
        column = mapping["source_column"]

        if column not in df.columns:
            coverage = 0.0
            n_valid = 0
        else:
            values = pd.to_numeric(
                df[column],
                errors="coerce"
            )

            n_valid = int(
                values.notna().sum()
            )

            coverage = (
                n_valid / len(df)
                if len(df) > 0
                else np.nan
            )

        coverage_records.append({
            "source_dataset": source,
            "model_feature": feature,
            "source_column": column,
            "rows": len(df),
            "n_valid": n_valid,
            "coverage_ratio": coverage
        })

source_feature_coverage = pd.DataFrame(
    coverage_records
)

display(
    source_feature_coverage
)


,source_dataset,model_feature,source_column,rows,n_valid,coverage_ratio
0,Catching_UX_Flaws,structure_depth,static_max_dom_depth,2,2,1.000
1,Catching_UX_Flaws,interactive_control_count,static_total_interactive_controls,2,2,1.000
2,Catching_UX_Flaws,interactive_label_ratio,static_control_label_proxy_ratio,2,2,1.000
3,UICrit_RICO,structure_depth,rico_max_tree_depth,1000,1000,1.000
4,UICrit_RICO,interactive_control_count,rico_interactive_node_count,1000,1000,1.000
5,UICrit_RICO,interactive_label_ratio,rico_interactive_label_proxy_ratio,1000,998,0.998


## 12. Elegibilidad preliminar de fuentes

In [13]:
def has_targets(df):
    target_cols = [
        "H1_violation",
        "H4_violation",
        "H5_violation",
        "H6_violation"
    ]

    existing = [
        col
        for col in target_cols
        if col in df.columns
    ]

    if not existing:
        return False, 0

    total_labeled = int(
        df[
            existing
        ]
        .notna()
        .sum()
        .sum()
    )

    return total_labeled > 0, total_labeled


catching_has_y, catching_y_count = has_targets(
    catching
)

rico_has_y, rico_y_count = has_targets(
    rico
)

row_eligibility = pd.DataFrame([
    {
        "source_dataset": "Catching_UX_Flaws",
        "rows": len(catching),
        "observed_nielsen_y_available": catching_has_y,
        "observed_label_cells": catching_y_count,
        "strict_feature_space_available": len(strict_feature_names) > 0,
        "proxy_feature_space_available": all(
            source_feature_coverage[
                source_feature_coverage[
                    "source_dataset"
                ]
                ==
                "Catching_UX_Flaws"
            ][
                "coverage_ratio"
            ]
            >
            0
        ),
        "candidate_role_strict": (
            "L"
            if (
                catching_has_y
                and len(strict_feature_names) > 0
            )
            else "NOT_ELIGIBLE"
        ),
        "candidate_role_proxy": (
            "L"
            if catching_has_y
            else "U"
        )
    },
    {
        "source_dataset": "UICrit_RICO",
        "rows": len(rico),
        "observed_nielsen_y_available": False,
        "observed_label_cells": 0,
        "strict_feature_space_available": len(strict_feature_names) > 0,
        "proxy_feature_space_available": all(
            source_feature_coverage[
                source_feature_coverage[
                    "source_dataset"
                ]
                ==
                "UICrit_RICO"
            ][
                "coverage_ratio"
            ]
            >
            0
        ),
        "candidate_role_strict": (
            "U"
            if len(strict_feature_names) > 0
            else "NOT_ELIGIBLE"
        ),
        "candidate_role_proxy": "U"
    },
    {
        "source_dataset": "29042675",
        "rows": np.nan,
        "observed_nielsen_y_available": True,
        "observed_label_cells": np.nan,
        "strict_feature_space_available": False,
        "proxy_feature_space_available": False,
        "candidate_role_strict": "NOT_ELIGIBLE_PENDING_X",
        "candidate_role_proxy": "NOT_ELIGIBLE_PENDING_X"
    },
    {
        "source_dataset": "Dashboards",
        "rows": np.nan,
        "observed_nielsen_y_available": False,
        "observed_label_cells": 0,
        "strict_feature_space_available": False,
        "proxy_feature_space_available": False,
        "candidate_role_strict": "PENDING_ROW_FEATURE_DATA",
        "candidate_role_proxy": "PENDING_ROW_FEATURE_DATA"
    },
    {
        "source_dataset": "A11y_CUA",
        "rows": np.nan,
        "observed_nielsen_y_available": False,
        "observed_label_cells": 0,
        "strict_feature_space_available": False,
        "proxy_feature_space_available": False,
        "candidate_role_strict": "PENDING_ROW_FEATURE_DATA",
        "candidate_role_proxy": "PENDING_ROW_FEATURE_DATA"
    }
])

display(
    row_eligibility
)


,source_dataset,rows,observed_nielsen_y_available,observed_label_cells,strict_feature_space_available,proxy_feature_space_available,candidate_role_strict,candidate_role_proxy
0,Catching_UX_Flaws,2.0,True,8.0,False,True,NOT_ELIGIBLE,L
1,UICrit_RICO,1000.0,False,0.0,False,True,NOT_ELIGIBLE,U
2,29042675,NaN,True,NaN,False,False,NOT_ELIGIBLE_PENDING_X,NOT_ELIGIBLE_PENDING_X
3,Dashboards,NaN,False,0.0,False,False,PENDING_ROW_FEATURE_DATA,PENDING_ROW_FEATURE_DATA
4,A11y_CUA,NaN,False,0.0,False,False,PENDING_ROW_FEATURE_DATA,PENDING_ROW_FEATURE_DATA


## 13. Verificar diversidad de las etiquetas en Catching

El hecho de tener `L` no garantiza que un clasificador binario sea entrenable.  
Se comprueba 0/1 por heurística sin inventar negativos.


In [14]:
target_balance_records = []

for target in [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]:
    if target not in catching.columns:
        target_balance_records.append({
            "target": target,
            "n_labeled": 0,
            "n_0": 0,
            "n_1": 0,
            "n_classes": 0
        })
        continue

    values = pd.to_numeric(
        catching[target],
        errors="coerce"
    )

    target_balance_records.append({
        "target": target,
        "n_labeled": int(values.notna().sum()),
        "n_0": int((values == 0).sum()),
        "n_1": int((values == 1).sum()),
        "n_classes": int(values.dropna().nunique())
    })

catching_target_balance = pd.DataFrame(
    target_balance_records
)

display(
    catching_target_balance
)


,target,n_labeled,n_0,n_1,n_classes
0,H1_violation,2,0,2,1
1,H4_violation,2,0,2,1
2,H5_violation,2,0,2,1
3,H6_violation,2,0,2,1


## 14. Decisión del espacio semisupervisado

La decisión separa dos preguntas:

### A. ¿Existe espacio canónico estricto?
Requiere equivalencia exacta/derivable exacta.

### B. ¿Existe un espacio proxy común para experimentación?
Puede existir aunque A sea no.

Incluso si B existe, el entrenamiento semisupervisado convencional seguirá requiriendo una auditoría de diversidad de clases en el notebook 27.


In [15]:
strict_available = (
    len(strict_feature_names) > 0
)

proxy_available = (
    len(usable_proxy_features) > 0
)

all_targets_two_classes = bool(
    (
        catching_target_balance[
            "n_classes"
        ]
        >= 2
    ).all()
)

if strict_available:
    primary_feature_decision = (
        "STRICT_SPACE_AVAILABLE"
    )
else:
    primary_feature_decision = (
        "NO_STRICT_COMMON_FEATURE_SPACE"
    )

if proxy_available:
    proxy_feature_decision = (
        "PROXY_SPACE_AVAILABLE_FOR_EXPERIMENT"
    )
else:
    proxy_feature_decision = (
        "NO_PROXY_COMMON_FEATURE_SPACE"
    )

feature_space_decision = pd.DataFrame([
    {
        "decision": "STRICT_COMMON_X_AVAILABLE",
        "value": strict_available,
        "detail": primary_feature_decision
    },
    {
        "decision": "EXPERIMENTAL_PROXY_X_AVAILABLE",
        "value": proxy_available,
        "detail": proxy_feature_decision
    },
    {
        "decision": "CATCHING_HAS_TWO_CLASSES_ALL_TARGETS",
        "value": all_targets_two_classes,
        "detail": (
            "Required for ordinary binary supervised seed; false does not automatically invalidate PU-learning research."
        )
    },
    {
        "decision": "EXTENSION_CAN_DERIVE_PROXY_FEATURES",
        "value": proxy_available,
        "detail": (
            "The future extension can compute analogous DOM structural constructs, but Android/Web domain shift must remain explicit."
        )
    }
])

display(
    feature_space_decision
)


,decision,value,detail
0,STRICT_COMMON_X_AVAILABLE,False,NO_STRICT_COMMON_FEATURE_SPACE
1,EXPERIMENTAL_PROXY_X_AVAILABLE,True,PROXY_SPACE_AVAILABLE_FOR_EXPERIMENT
2,CATCHING_HAS_TWO_CLASSES_ALL_TARGETS,False,Required for ordinary binary supervised seed; false does not automatically invalidate PU-learning research.
3,EXTENSION_CAN_DERIVE_PROXY_FEATURES,True,"The future extension can compute analogous DOM structural constructs, but Android/Web domain shift must remain explicit."


## 15. Definir archivo de feature space

In [16]:
feature_space_rows = []

for feature in strict_feature_names:
    feature_space_rows.append({
        "feature_space": "STRICT_CANONICAL",
        "model_feature": feature,
        "scientific_role": "PRIMARY_CANDIDATE",
        "mapping_level": "EXACT_OR_DERIVABLE_EXACT",
        "extension_derivable": True
    })

for feature in usable_proxy_features:
    feature_space_rows.append({
        "feature_space": "EXPERIMENTAL_CONSTRUCT_PROXY",
        "model_feature": feature,
        "scientific_role": "SECONDARY_EXPERIMENT_ONLY",
        "mapping_level": "CONSTRUCT_PROXY",
        "extension_derivable": True
    })

semisupervised_feature_space = pd.DataFrame(
    feature_space_rows,
    columns=[
        "feature_space",
        "model_feature",
        "scientific_role",
        "mapping_level",
        "extension_derivable"
    ]
)

display(
    semisupervised_feature_space
)


,feature_space,model_feature,scientific_role,mapping_level,extension_derivable
0,EXPERIMENTAL_CONSTRUCT_PROXY,structure_depth,SECONDARY_EXPERIMENT_ONLY,CONSTRUCT_PROXY,True
1,EXPERIMENTAL_CONSTRUCT_PROXY,interactive_control_count,SECONDARY_EXPERIMENT_ONLY,CONSTRUCT_PROXY,True
2,EXPERIMENTAL_CONSTRUCT_PROXY,interactive_label_ratio,SECONDARY_EXPERIMENT_ONLY,CONSTRUCT_PROXY,True


## 16. Guardar resultados

In [17]:
proxy_mapping_audit.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_feature_mapping.csv",
    index=False
)

semisupervised_feature_space.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_feature_space.csv",
    index=False
)

source_feature_coverage.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_source_feature_coverage.csv",
    index=False
)

row_eligibility.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_row_eligibility_by_source.csv",
    index=False
)

schema_audit.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_source_schema_audit.csv",
    index=False
)

proxy_feature_validation.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_proxy_feature_validation.csv",
    index=False
)

catching_target_balance.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_catching_target_balance.csv",
    index=False
)

feature_space_decision.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_feature_space_decision.csv",
    index=False
)

feature_space_registry.to_csv(
    SEMISUP_REPORTS_DIR
    / "semisupervised_feature_space_registry.csv",
    index=False
)


In [18]:
expected_outputs = [
    "semisupervised_feature_mapping.csv",
    "semisupervised_feature_space.csv",
    "semisupervised_source_feature_coverage.csv",
    "semisupervised_row_eligibility_by_source.csv",
    "semisupervised_source_schema_audit.csv",
    "semisupervised_proxy_feature_validation.csv",
    "semisupervised_catching_target_balance.csv",
    "semisupervised_feature_space_decision.csv",
    "semisupervised_feature_space_registry.csv"
]

for filename in expected_outputs:
    path = (
        SEMISUP_REPORTS_DIR
        /
        filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )


semisupervised_feature_mapping.csv ✅
semisupervised_feature_space.csv ✅
semisupervised_source_feature_coverage.csv ✅
semisupervised_row_eligibility_by_source.csv ✅
semisupervised_source_schema_audit.csv ✅
semisupervised_proxy_feature_validation.csv ✅
semisupervised_catching_target_balance.csv ✅
semisupervised_feature_space_decision.csv ✅
semisupervised_feature_space_registry.csv ✅


## Punto de detención

No construir todavía `L ∪ U` manualmente.

El notebook 26 utilizará **solo los resultados ejecutados de este notebook** para decidir qué espacio puede materializarse:

`26_semisupervised_labeled_unlabeled_split.ipynb`

Dos escenarios son posibles:

### Escenario 1 — `STRICT_COMMON_X_AVAILABLE=True`
Se construirá el dataset semisupervisado principal sobre ese espacio.

### Escenario 2 — strict=False y proxy=True
Se podrá materializar un dataset `L/U` **experimental** sobre el espacio estructural proxy, manteniendo explícita la diferencia Web DOM ↔ Android UI hierarchy.

Además, si el conjunto `L` continúa conteniendo una sola clase, el notebook 27 deberá distinguir:

- SSL binario convencional;
- Positive–Unlabeled (PU) learning;
- o `NOT_TRAINABLE`.

No se crearán negativos artificiales.
